# Publish your DLM to Hugging Face, then run it anywhere

**Google Drive `resume.pt` → inference bundle → Hugging Face Hub → download and generate.**

### First upload
1. Set `RESUME_PATH` to your checkpoint's full Drive path and `HF_REPO_ID` to `your-username/model-name` in cell 1.
2. In Colab's **Secrets** (key icon), add `HF_TOKEN` with a Hugging Face token that can write to that repository, and enable notebook access. Create a token at https://huggingface.co/settings/tokens.
3. Choose the repository visibility and model-card license, then **Runtime → Run all**. Uploading works on CPU; a GPU speeds up generation.

### Later, in a fresh Colab session
Set `MODE = "inference"` and the same `HF_REPO_ID`, then run all. This mode downloads the weights, tokenizer, and inference code from HF. A private repository needs a token with read access.

**Checkpoint compatibility:** this notebook supports `resume.pt` from this project's **`run_colab.ipynb` / `dlm/train.py`** (the RMSNorm/RoPE/SwiGLU model). Earlier standalone `Final_Diffusion_...` notebooks use another architecture and need their matching exporter.

The bundle includes `model.safetensors`, model and generation configuration, the original tokenizer, `dlm/`, `infer.py`, and a model card. The recorded linear/cosine schedule is preserved. The base model generates or continues short Stories. Loading uses the bundled custom sampler; `AutoModelForCausalLM` is not supported. Hosting a live API is a separate deployment.

In [ ]:
# 1. Settings
MODE = "publish" #@param ["publish", "inference"]
RESUME_PATH = "/content/drive/MyDrive/diffusion_lm_outputs/checkpoints/pretrain_cosine/resume.pt" #@param {type:"string"}
HF_REPO_ID = "" #@param {type:"string"}
HF_PRIVATE = True #@param {type:"boolean"}
HF_LICENSE = "apache-2.0" #@param {type:"string"}
TOKENIZER_DIR = "" #@param {type:"string"}
HF_REVISION = "main" #@param {type:"string"}

# TOKENIZER_DIR: original tokenizer folder, if you trained with a custom one.
# Blank: look beside the run, in final/tokenizer or best/tokenizer, then in the code repo.
# run_colab.ipynb uses the committed repo tokenizer; resume.pt itself does not contain it.
# HF_PRIVATE applies when creating a repo; an existing repo keeps its HF visibility setting.
# HF_REVISION is for inference mode: main, a tag, or a specific HF commit.
# Publish mode verifies the exact commit returned by the upload.

import re
if MODE not in ("publish", "inference"):
    raise ValueError("MODE must be publish or inference.")
HF_REPO_ID = HF_REPO_ID.strip()
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*/[A-Za-z0-9][A-Za-z0-9._-]*", HF_REPO_ID):
    raise ValueError("Set HF_REPO_ID to your-username/model-name, for example alice/dlm-tinystories.")
if MODE == "publish" and (not RESUME_PATH.strip() or not HF_LICENSE.strip()):
    raise ValueError("Set RESUME_PATH and HF_LICENSE before publishing.")

In [ ]:
# 2. Install missing inference dependencies; get exporter code only in publish mode
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

missing = [p for p in ("tokenizers", "safetensors", "huggingface_hub")
           if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import torch  # Use Colab's preinstalled torch and numpy.
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

SOURCE_REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
REPO_DIR = Path("/content/dlm_hf_source")
if MODE == "publish":
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", SOURCE_REPO, str(REPO_DIR)], check=True)
    CODE_COMMIT = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    if "dlm" in sys.modules and Path(sys.modules["dlm"].__file__).resolve().parent != REPO_DIR / "dlm":
        raise RuntimeError("Another DLM copy is loaded. Restart the runtime and run this notebook again.")
    sys.path.insert(0, str(REPO_DIR))
    print("Exporter source commit:", CODE_COMMIT)
else:
    print("Inference mode: weights and code will come from Hugging Face.")

In [ ]:
# 3. Authentication (the token is read from Secrets or a cached login)
from huggingface_hub import HfApi, get_token, snapshot_download

_hf_token = get_token()
if not _hf_token:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
    except Exception:
        _hf_token = None
if MODE == "publish" and not _hf_token:
    raise RuntimeError("Add a write-access HF_TOKEN in Colab Secrets, enable notebook access, and rerun this cell.")
api = HfApi(token=_hf_token)
print("Authentication available." if _hf_token else "Downloading as an anonymous user (public repositories only).")

In [ ]:
# 4. Mount Drive and export the selected resume.pt into a temporary local folder
if MODE == "publish":
    import tempfile
    from google.colab import drive
    from tokenizers import Tokenizer
    from dlm.train import export_checkpoint

    drive.mount("/content/drive")
    source = Path(RESUME_PATH).expanduser()
    if not source.is_file():
        raise FileNotFoundError(f"No checkpoint at {source}. Copy the full path from Colab's Drive file browser into RESUME_PATH.")
    candidates = ([Path(TOKENIZER_DIR).expanduser()] if TOKENIZER_DIR.strip() else [
        source.parent / "tokenizer", source.parent / "final" / "tokenizer",
        source.parent / "best" / "tokenizer", REPO_DIR / "tokenizer"])
    tokenizer_dir = next((p for p in candidates if (p / "tokenizer.json").is_file()), None)
    if tokenizer_dir is None:
        raise FileNotFoundError("Set TOKENIZER_DIR to the original tokenizer folder containing tokenizer.json.")

    WORK_DIR = Path(tempfile.mkdtemp(prefix="dlm-hf-"))
    STAGED_RUN = WORK_DIR / "run"
    try:
        # resume.pt is your own training checkpoint, including optimizer/RNG objects.
        exported = export_checkpoint(source, out=STAGED_RUN / "final", tokenizer_dir=tokenizer_dir)
    except (KeyError, TypeError, ValueError) as e:
        raise ValueError("This checkpoint does not match dlm/train.py. Older standalone notebooks need their original model/export code.") from e
    cfg = json.loads((exported / "config.json").read_text())
    tokenizer = Tokenizer.from_file(str(exported / "tokenizer" / "tokenizer.json"))
    if tokenizer.get_vocab_size() != cfg["vocab_size"]:
        raise ValueError("Tokenizer vocabulary size differs from the model. Set TOKENIZER_DIR to the training tokenizer.")
    special_names = ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]", "<|user|>", "<|assistant|>", "<|system|>", "<|end|>"]
    if any(tokenizer.token_to_id(name) != i for i, name in enumerate(special_names)):
        raise ValueError("Tokenizer special-token IDs differ from this project's training format.")
    if cfg["pad_id"] != tokenizer.token_to_id("[PAD]") or cfg["mask_id"] != tokenizer.token_to_id("[MASK]"):
        raise ValueError("Model and tokenizer PAD/MASK IDs differ.")
    export_info = json.loads((exported / "export.json").read_text())
    run_info = json.loads((exported / "run_info.json").read_text())
    SCHEDULE = run_info["recipe"]["schedule"]
    if SCHEDULE not in ("linear", "cosine"):
        raise ValueError(f"Unsupported checkpoint schedule: {SCHEDULE}")
    print(f"Exported step {export_info['step']:,} / {export_info['steps_planned']:,}; schedule: {SCHEDULE}")
    print("Tokenizer:", tokenizer_dir)
    print("Local export:", exported)
else:
    print("Using the HF bundle; Drive export is skipped.")

In [ ]:
# 5. Portable inference entry point, included in the HF bundle as infer.py
INFERENCE_SCRIPT = r'''"""Run the bundled DLM: python infer.py --prompt 'Once upon a time'"""
import argparse
import json
import sys
from pathlib import Path

# HF cache files can be symlinks into blobs/. Keep the snapshot directory.
BUNDLE_DIR = Path(__file__).absolute().parent
sys.path.insert(0, str(BUNDLE_DIR))

from dlm.sample import generate_text, load_checkpoint
from dlm.train import pick_device


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--model-dir', type=Path, default=BUNDLE_DIR)
    p.add_argument('--prompt', default='')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--max-new-tokens', type=int, default=None)
    p.add_argument('--temperature', type=float, default=None)
    p.add_argument('--block-length', type=int, default=None)
    p.add_argument('--steps', type=int, default=None)
    p.add_argument('--schedule', choices=('auto', 'linear', 'cosine'), default='auto')
    p.add_argument('--device', choices=('auto', 'cpu', 'cuda', 'mps'), default='auto')
    a = p.parse_args()
    saved = json.loads((a.model_dir / 'generation_config.json').read_text())
    settings = {key: saved[key] for key in ('max_new_tokens', 'temperature', 'block_length', 'schedule')}
    for key in ('max_new_tokens', 'temperature', 'block_length'):
        if getattr(a, key) is not None:
            settings[key] = getattr(a, key)
    if a.schedule != 'auto':
        settings['schedule'] = a.schedule
    model, tok = load_checkpoint(a.model_dir, device=pick_device(a.device))
    model.eval()
    result = generate_text(model, tok, a.prompt, seed=a.seed, steps=a.steps, **settings)
    print(result['text'])
    print(f"\n[{result['forward_calls']} forward calls; schedule={settings['schedule']}; truncated={result['truncated']}]")
    for warning in result['warnings']:
        print(warning)


if __name__ == '__main__':
    main()
'''

In [ ]:
# 6. Build and upload the inference bundle
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

UPLOADED_REVISION = None
if MODE == "publish":
    import shutil
    from dlm.hub import MARKER, prepare_folder
    from dlm.sample import load_checkpoint

    UPLOAD_DIR = WORK_DIR / "upload"
    prepare_folder(STAGED_RUN, UPLOAD_DIR, HF_REPO_ID, license=HF_LICENSE)
    generation = json.loads((UPLOAD_DIR / "generation_config.json").read_text())
    generation["schedule"] = SCHEDULE
    (UPLOAD_DIR / "generation_config.json").write_text(json.dumps(generation, indent=2) + "\n")
    # Include the exact Python source used for export so future inference needs only HF.
    (UPLOAD_DIR / "dlm").mkdir(exist_ok=True)
    for py in sorted((REPO_DIR / "dlm").glob("*.py")):
        shutil.copy2(py, UPLOAD_DIR / "dlm" / py.name)
    (UPLOAD_DIR / "infer.py").write_text(INFERENCE_SCRIPT)
    (UPLOAD_DIR / "requirements-inference.txt").write_text("torch\nnumpy\ntokenizers\nsafetensors\nhuggingface_hub\n")
    # Replace the generic usage section with instructions for this self-contained bundle.
    card_path = UPLOAD_DIR / "README.md"
    card = card_path.read_text()
    before, rest = card.split("## How to use", 1)
    _, references = rest.split("## References", 1)
    usage = f'''## How to use

This bundle includes its custom PyTorch DLM and diffusion sampler in `dlm/`.
Use Python 3.10+ with `pip install torch numpy tokenizers safetensors huggingface_hub`.
For a private repository, first run `hf auth login` with a read-access token.

```python
import subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download

folder = Path(snapshot_download({HF_REPO_ID!r}))
subprocess.run([sys.executable, str(folder / 'infer.py'),
                '--prompt', 'Once upon a time, there was a little rabbit',
                '--max-new-tokens', '128'], check=True)
```

The default sampling schedule is **{SCHEDULE}**, read from `generation_config.json`.
`--device auto` selects CUDA, MPS, or CPU. Omit the prompt for an unconditional Story.
`--temperature 0` uses greedy decoding. Use `revision='<HF commit>'` with
`snapshot_download` to pin the model and code together.

This is not a Transformers `AutoModelForCausalLM` model or a hosted inference API.
Exported checkpoint step: **{export_info['step']} / {export_info['steps_planned']}**.
Source code commit: `{CODE_COMMIT}`.

'''
    card = before + usage + "## References" + references
    if not (UPLOAD_DIR / "training_curves.png").is_file():
        card = card.replace("![training curves](training_curves.png)\n", "")
    card_path.write_text(card)
    # Reload safetensors and the tokenizer before starting the network upload.
    checked_model, checked_tok = load_checkpoint(UPLOAD_DIR, device="cpu")
    print(f"Validated {checked_model.num_parameters():,} parameters; {checked_tok.get_vocab_size():,} tokens.")
    del checked_model, checked_tok
    expected_hashes = {str(p.relative_to(UPLOAD_DIR)): sha256_file(p)
                       for p in sorted(UPLOAD_DIR.rglob("*")) if p.is_file() and p.name != MARKER}
    publication = {"repo_id": HF_REPO_ID, "checkpoint_step": export_info["step"],
                   "steps_planned": export_info["steps_planned"], "schedule": SCHEDULE,
                   "source_repository": SOURCE_REPO, "source_commit": CODE_COMMIT,
                   "files_sha256": expected_hashes}
    (UPLOAD_DIR / "publication.json").write_text(json.dumps(publication, indent=2) + "\n")
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=HF_PRIVATE, exist_ok=True)
    commit = api.upload_folder(
        repo_id=HF_REPO_ID, repo_type="model", folder_path=str(UPLOAD_DIR),
        ignore_patterns=[MARKER],
        commit_message=f"Publish DLM checkpoint step {export_info['step']} with portable inference code")
    UPLOADED_REVISION = commit.oid
    print(f"Uploaded: https://huggingface.co/{HF_REPO_ID}")
    print("HF commit:", UPLOADED_REVISION)
else:
    print("Inference mode: upload skipped.")

In [ ]:
# 7. Download from HF and verify the bundle's file hashes
if MODE == "publish" and not UPLOADED_REVISION:
    raise RuntimeError("Complete the upload cell before verifying the published model.")
revision = UPLOADED_REVISION if MODE == "publish" else HF_REVISION
SNAPSHOT_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, repo_type="model",
                                      revision=revision, token=_hf_token))
if not (SNAPSHOT_DIR / "infer.py").is_file() or not (SNAPSHOT_DIR / "publication.json").is_file():
    raise RuntimeError("This HF revision lacks the portable bundle. Publish it once with this notebook first.")
downloaded_publication = json.loads((SNAPSHOT_DIR / "publication.json").read_text())
hashes = expected_hashes if MODE == "publish" else downloaded_publication["files_sha256"]
for name, expected in hashes.items():
    path = SNAPSHOT_DIR / name
    if not path.is_file() or sha256_file(path) != expected:
        raise RuntimeError(f"Downloaded bundle failed verification: {name}")
print(f"Verified {len(hashes)} files from HF revision {revision}.")
print("Downloaded snapshot:", SNAPSHOT_DIR)

In [ ]:
# 8. Generate using ONLY the downloaded bundle (rerun with your own opening words)
PROMPT = "Once upon a time, there was a little rabbit" #@param {type:"string"}
MAX_NEW_TOKENS = 128 #@param {type:"integer"}
TEMPERATURE = 0.8 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
STEPS = 0 #@param {type:"integer"}
DEVICE = "auto" #@param ["auto", "cpu", "cuda", "mps"]
# STEPS = 0 uses the sampler default. A positive value sets a denoising-step budget.
if MAX_NEW_TOKENS < 1 or TEMPERATURE < 0 or STEPS < 0:
    raise ValueError("Use MAX_NEW_TOKENS >= 1, TEMPERATURE >= 0, and STEPS >= 0.")
command = [sys.executable, str(SNAPSHOT_DIR / "infer.py"),
           "--prompt", PROMPT, "--max-new-tokens", str(MAX_NEW_TOKENS),
           "--temperature", str(TEMPERATURE), "--seed", str(SEED), "--device", DEVICE]
if STEPS > 0:
    command += ["--steps", str(STEPS)]
# A fresh process ensures imports come from the downloaded dlm/, not the exporter in memory.
subprocess.run(command, cwd=SNAPSHOT_DIR, check=True)
print("\nHF download → inference succeeded.")
if MODE == "publish":
    receipt = {"repo_id": HF_REPO_ID, "revision": UPLOADED_REVISION,
               "checkpoint_step": export_info["step"], "schedule": SCHEDULE,
               "download_and_inference_verified": True}
    receipt_path = source.parent / "hf_publication.json"
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n")
    print("Publication receipt saved to Drive:", receipt_path)

## Use the published model on another machine

Install **Python 3.10+** and `torch numpy tokenizers safetensors huggingface_hub`. For a private model, run `hf auth login` with a read-access token. Run the Python snippet printed below; it pins the HF revision so weights and code stay together.

For another Colab session, set `MODE = "inference"`, fill in `HF_REPO_ID`, and optionally set `HF_REVISION` to the printed commit.

When you finish experimenting in Colab, choose **Runtime → Disconnect and delete runtime**.

In [ ]:
# 9. Copy this Python snippet to your laptop, another Colab, or an AWS machine
# snapshot_download's cache directory is named after the resolved HF commit.
resolved_revision = UPLOADED_REVISION if MODE == "publish" else SNAPSHOT_DIR.name
print(f'''import subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download

folder = Path(snapshot_download(
    repo_id={HF_REPO_ID!r},
    revision={resolved_revision!r},
))
subprocess.run([sys.executable, str(folder / "infer.py"),
                "--prompt", "Once upon a time, there was a little rabbit",
                "--max-new-tokens", "128", "--device", "auto"], check=True)
''')